In [ ]:
import json
import pandas as pd

# 1. โหลดข้อมูลสถิติการเลือกตั้งและข้อมูลชื่อจังหวัด
with open('data/stats_cons.json', 'r', encoding='utf-8') as f:
    stats_data = json.load(f)

with open('data/info_province.json', 'r', encoding='utf-8') as f:
    province_info = json.load(f)

# 2. สร้าง Mapping รหัสจังหวัดเป็นชื่อจังหวัด
prov_map = {p['prov_id']: p['province'] for p in province_info['province']}

# 3. ประมวลผลและสรุปข้อมูลรายจังหวัด
election_summary = []

for prov in stats_data['result_province']:
    p_id = prov['prov_id']
    p_name = prov_map.get(p_id, p_id)
    
    election_summary.append({
        'จังหวัด': p_name,
        'ผู้มาใช้สิทธิ (เขต)': prov.get('turn_out', 0),
        'บัตรเสีย (เขต)': prov.get('invalid_votes', 0),
        'ไม่ประสงค์ลงคะแนน (เขต)': prov.get('blank_votes', 0),
        'ผู้มาใช้สิทธิ (บัญชีรายชื่อ)': prov.get('party_list_turn_out', 0),
        'บัตรเสีย (บัญชีรายชื่อ)': prov.get('party_list_invalid_votes', 0),
        'ไม่ประสงค์ลงคะแนน (บัญชีรายชื่อ)': prov.get('party_list_blank_votes', 0),
        'ผลต่างแบบเขต-บัญชีรายชื่อ)': abs(prov.get('turn_out', 0) - prov.get('party_list_turn_out', 0))
    })

# 4. แปลงเป็น DataFrame เพื่อการแสดงผลและจัดเรียงตามชื่อจังหวัด
df = pd.DataFrame(election_summary)
df = df.sort_values(by='ผลต่างแบบเขต-บัญชีรายชื่อ)', ascending=False)

# 5. แสดงผลลัพธ์ (ตัวอย่าง 15 จังหวัดแรก)
print("สรุปสถิติการเลือกตั้งแยกตามจังหวัด:")
print(df.head(15).to_string(index=False))

# 6. บันทึกเป็นไฟล์ CSV เพื่อนำไปใช้งานต่อ
df.to_csv('output/election_summary_by_province.csv', index=False, encoding='utf-8-sig')
print("\nบันทึกข้อมูลลงไฟล์ 'election_summary_by_province.csv' เรียบร้อยแล้ว")